## Neural Network Models — Classification (`Demand_Category`)

Same evaluation harness as `logistic_regression.ipynb` / `tree_models.ipynb`:

- Same 4 base datasets: `train`, `train_preserved_imputed`, `train_clean_imputed`, `train_clean`
- Same 5 feature-engineering stages (baseline → skew transforms → weather categories → +Weekday → cyclic time)
- Same `StratifiedKFold(5)` + macro-F1 evaluation

Architectures are pulled from the regression project's `nn_models.ipynb`: **Basic Tabular NN, Entity Embedding NN,
FT-Transformer** — same three model definitions, adapted from single-output `log1p`-regression (`MSELoss`, 1 output
unit) to 3-class classification (`CrossEntropyLoss`, 3 output logits over `Demand_Category` ∈ {0=Low, 1=Normal, 2=High}).

Two deliberate fixes vs. the regression source, both purely additive (no behavior removed):

1. **Device selection now also checks Apple Metal (MPS)** — the regression notebook only checked
   `torch.cuda.is_available()`, so it silently ran CPU-only on Apple Silicon. This is a fresh file, not an edit of
   an existing tracked notebook, so the fix is applied directly here: `cuda` → `mps` → `cpu`.
2. **Per-fold, leak-free imputation** instead of hard-failing on NaN. The regression version raised a `ValueError`
   if any missing values reached `evaluate_nn_architectures` and required pre-imputed input. Here, `IterativeImputer`
   / `SimpleImputer` are fit on the training fold only and applied to the validation fold — consistent with
   `build_preprocessor()` in `logistic_regression.ipynb`, and avoiding the imputation leakage flagged in that
   notebook's pre-imputed CSV inputs. This also means the raw `train` dataset (which still has real NaNs) does not
   need a separate pre-imputed file.

A `QUICK_MODE` toggle is included below — the full grid (4 datasets × 5 steps × 3 architectures × 5 folds) is a large
CPU/GPU-time budget; flip it on for a fast smoke test before committing to a full run.


In [ ]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.experimental import enable_iterative_imputer  # noqa: F401  (required to unlock IterativeImputer)
from sklearn.impute import IterativeImputer, SimpleImputer

df_train = pd.read_csv("train.csv")
df_train_preserved_imputed = pd.read_csv("train_preserved_imputed.csv")
df_train_clean_imputed = pd.read_csv("train_clean_imputed.csv")
df_train_clean = pd.read_csv("train_clean.csv")


In [ ]:
# ============================================================
# COMMON CONFIGURATION
# ============================================================

TARGET = "Demand_Category"
NUM_CLASSES = 3

DATASETS = {
    "train": df_train.copy(),
    "train_preserved_imputed": df_train_preserved_imputed.copy(),
    "train_clean_imputed": df_train_clean_imputed.copy(),
    "train_clean": df_train_clean.copy(),
}

BASE_NUMERIC_FEATURES = [
    "Hour",
    "Temperature",
    "Humidity",
    "Wind speed",
    "Visibility",
    "Dew point temperature",
    "Solar Radiation",
    "Rainfall",
    "Snowfall",
]

BASE_CATEGORICAL_FEATURES = [
    "Seasons",
    "Holiday",
    "Functioning Day",
]

CV = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

NN_MODELS = ["Basic Tabular NN", "Entity Embedding NN", "FT-Transformer"]

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using device: {DEVICE}")

# Flip to True for a fast smoke test (1 dataset, first 2 steps, few epochs)
# before committing to the full grid below.
QUICK_MODE = False
EPOCHS = 8 if QUICK_MODE else 25
BATCH_SIZE = 256


In [ ]:
# ============================================================
# FEATURE ENGINEERING (identical to logistic_regression.ipynb / tree_models.ipynb)
# ============================================================

def add_skew_transformations(df):
    """Add transformed versions of skewed weather variables. Originals retained."""

    out = df.copy()

    right_skewed = ["Wind speed", "Solar Radiation", "Rainfall", "Snowfall"]

    for col in right_skewed:
        if col in out.columns:
            values = out[col].mask(out[col] < 0)
            out[f"{col}_log1p"] = np.log1p(values)

    if "Visibility" in out.columns:
        values = out["Visibility"].mask(out["Visibility"] < 0)
        out["Visibility_pow3"] = values ** 3

    return out


def add_weather_categories(df):
    """Add categorical versions of heavily skewed weather variables."""

    out = df.copy()

    if "Visibility" in out.columns:
        bins = [-np.inf, 500, 1500, np.inf]
        out["Visibility_Cat"] = pd.cut(
            out["Visibility"], bins=bins, labels=["low", "medium", "high"]
        )

    if "Solar Radiation" in out.columns:
        bins = [-np.inf, 0.1, 1.5, np.inf]
        out["Solar_Radiation_Cat"] = pd.cut(
            out["Solar Radiation"], bins=bins, labels=["none", "low", "high"]
        )

    if "Rainfall" in out.columns:
        bins = [-np.inf, 0.1, 2.0, np.inf]
        out["Rainfall_Cat"] = pd.cut(
            out["Rainfall"], bins=bins, labels=["none", "light", "heavy"]
        )

    if "Snowfall" in out.columns:
        bins = [-np.inf, 0.1, 1.0, np.inf]
        out["Snowfall_Cat"] = pd.cut(
            out["Snowfall"], bins=bins, labels=["none", "light", "heavy"]
        )

    return out


def add_weekday(df):
    """Add weekday derived from Date. Monday = 0, Sunday = 6."""

    out = df.copy()

    if "Date" in out.columns:
        date_parsed = pd.to_datetime(out["Date"], dayfirst=True, errors="coerce")
        out["Weekday"] = date_parsed.dt.dayofweek

    return out


def add_month(df):
    out = df.copy()

    if "Date" in out.columns:
        date_parsed = pd.to_datetime(out["Date"], dayfirst=True, errors="coerce")
        out["Month"] = date_parsed.dt.month

    return out


def add_all_cyclic_time_features(df):
    """Add cyclic representations for Hour, Weekday, Month, Seasons."""

    out = df.copy()

    if "Hour" in out.columns:
        out["Hour_sin"] = np.sin(2 * np.pi * out["Hour"] / 24)
        out["Hour_cos"] = np.cos(2 * np.pi * out["Hour"] / 24)

    if "Weekday" in out.columns:
        out["Weekday_sin"] = np.sin(2 * np.pi * out["Weekday"] / 7)
        out["Weekday_cos"] = np.cos(2 * np.pi * out["Weekday"] / 7)

    if "Month" in out.columns:
        out["Month_sin"] = np.sin(2 * np.pi * (out["Month"] - 1) / 12)
        out["Month_cos"] = np.cos(2 * np.pi * (out["Month"] - 1) / 12)

    if "Seasons" in out.columns:
        season_mapping = {"Winter": 0, "Spring": 1, "Summer": 2, "Autumn": 3}
        season_num = out["Seasons"].map(season_mapping)
        out["Season_sin"] = np.sin(2 * np.pi * season_num / 4)
        out["Season_cos"] = np.cos(2 * np.pi * season_num / 4)

    return out


SKEW_NUMERIC_FEATURES = BASE_NUMERIC_FEATURES + [
    "Wind speed_log1p",
    "Solar Radiation_log1p",
    "Rainfall_log1p",
    "Snowfall_log1p",
    "Visibility_pow3",
]

WEATHER_CATEGORY_FEATURES = BASE_CATEGORICAL_FEATURES + [
    "Visibility_Cat",
    "Solar_Radiation_Cat",
    "Rainfall_Cat",
    "Snowfall_Cat",
]


In [ ]:
# ============================================================
# ARCHITECTURES (from regression nn_models.ipynb, adapted to classification:
# output layer -> NUM_CLASSES logits, no log1p target transform)
# ============================================================

class UniversalTabularDataset(Dataset):
    def __init__(self, x_num, x_cat_ord, x_comb_ohe, y=None):
        self.x_num = torch.tensor(x_num, dtype=torch.float32)
        self.x_cat_ord = torch.tensor(x_cat_ord, dtype=torch.long)
        self.x_comb_ohe = torch.tensor(x_comb_ohe, dtype=torch.float32)

        self.y = None
        if y is not None:
            # Class indices (0/1/2) for CrossEntropyLoss -- not log1p transformed.
            self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.x_comb_ohe)

    def __getitem__(self, idx):
        y_val = self.y[idx] if self.y is not None else torch.tensor(-1)
        return self.x_num[idx], self.x_cat_ord[idx], self.x_comb_ohe[idx], y_val


class BasicTabularNN(nn.Module):
    def __init__(self, input_dim, num_classes=NUM_CLASSES):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, num_classes)
        )

    def forward(self, x_num, x_cat_ord, x_comb_ohe):
        # Only consumes the flat One-Hot vector, like the regression version.
        return self.network(x_comb_ohe)


class EntityEmbeddingNN(nn.Module):
    def __init__(self, num_numerical, cat_cardinalities, embedding_dim=16, num_classes=NUM_CLASSES):
        super().__init__()
        self.embeddings = nn.ModuleList([nn.Embedding(c, embedding_dim) for c in cat_cardinalities])

        input_dim = num_numerical + (len(cat_cardinalities) * embedding_dim)

        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, num_classes)
        )

    def forward(self, x_num, x_cat_ord, x_comb_ohe):
        emb_outputs = [emb(x_cat_ord[:, i]) for i, emb in enumerate(self.embeddings)]

        if emb_outputs and x_num.shape[1] > 0:
            x = torch.cat([x_num] + emb_outputs, dim=1)
        elif emb_outputs:
            x = torch.cat(emb_outputs, dim=1)
        else:
            x = x_num

        return self.network(x)


class FTTransformer(nn.Module):
    def __init__(self, num_numerical, cat_cardinalities, dim=64, depth=3, heads=4, dropout=0.2, num_classes=NUM_CLASSES):
        super().__init__()
        self.num_numerical = num_numerical

        self.num_embeddings = nn.ModuleList([nn.Linear(1, dim) for _ in range(num_numerical)])
        self.cat_embeddings = nn.ModuleList([nn.Embedding(c, dim) for c in cat_cardinalities])

        self.cls_token = nn.Parameter(torch.randn(1, 1, dim))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=dim, nhead=heads, dim_feedforward=dim * 4,
            dropout=dropout, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=depth)

        self.head = nn.Sequential(
            nn.LayerNorm(dim),
            nn.ReLU(),
            nn.Linear(dim, num_classes)
        )

    def forward(self, x_num, x_cat_ord, x_comb_ohe):
        batch_size = x_num.shape[0]
        tokens = []

        for i in range(self.num_numerical):
            tokens.append(self.num_embeddings[i](x_num[:, i:i + 1]).unsqueeze(1))

        for i, emb in enumerate(self.cat_embeddings):
            tokens.append(emb(x_cat_ord[:, i]).unsqueeze(1))

        x = torch.cat(tokens, dim=1) if tokens else torch.empty(batch_size, 0, self.cls_token.shape[-1]).to(x_num.device)

        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)

        x = self.transformer(x)
        return self.head(x[:, 0, :])


In [ ]:
def evaluate_nn_model(df, numeric_features, categorical_features, model_name, experiment_name="",
                       epochs=None, batch_size=None):
    """
    Evaluate a single NN architecture using the same Stratified 5-fold CV /
    macro-F1 harness as evaluate_logistic_regression() / evaluate_tree_model().

    Per fold: IterativeImputer (numeric) + SimpleImputer (categorical, most-frequent)
    are fit on the training fold only, then applied to the validation fold --
    a leak-free version of the imputation step used elsewhere in this project.
    """

    epochs = EPOCHS if epochs is None else epochs
    batch_size = BATCH_SIZE if batch_size is None else batch_size

    num_cols = [c for c in numeric_features if c in df.columns]
    cat_cols = [c for c in categorical_features if c in df.columns]

    X_num_raw = df[num_cols].values if num_cols else np.zeros((len(df), 0))
    X_cat_raw = df[cat_cols].astype(str).values if cat_cols else np.zeros((len(df), 0))
    y = df[TARGET].astype(int).values

    fold_scores = []

    for train_idx, val_idx in CV.split(X_num_raw if num_cols else X_cat_raw, y):

        # --- Leak-free imputation (fit on train fold only) ---
        if num_cols:
            num_imputer = IterativeImputer(max_iter=10, random_state=42)
            x_num_train_imp = num_imputer.fit_transform(X_num_raw[train_idx])
            x_num_val_imp = num_imputer.transform(X_num_raw[val_idx])
        else:
            x_num_train_imp = np.zeros((len(train_idx), 0))
            x_num_val_imp = np.zeros((len(val_idx), 0))

        if cat_cols:
            cat_imputer = SimpleImputer(strategy="most_frequent")
            x_cat_train_imp = cat_imputer.fit_transform(X_cat_raw[train_idx])
            x_cat_val_imp = cat_imputer.transform(X_cat_raw[val_idx])
        else:
            x_cat_train_imp = np.zeros((len(train_idx), 0))
            x_cat_val_imp = np.zeros((len(val_idx), 0))

        # --- Scaling / encoding (fit on train fold only) ---
        scaler = StandardScaler()
        x_num_train = scaler.fit_transform(x_num_train_imp) if num_cols else x_num_train_imp
        x_num_val = scaler.transform(x_num_val_imp) if num_cols else x_num_val_imp

        cat_cards = []
        if cat_cols:
            ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
            x_cat_ohe_train = ohe.fit_transform(x_cat_train_imp)
            x_cat_ohe_val = ohe.transform(x_cat_val_imp)

            oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
            x_cat_ord_train = oe.fit_transform(x_cat_train_imp).astype(int)
            x_cat_ord_val = oe.transform(x_cat_val_imp).astype(int)

            # Clip unknowns in validation to 0 to prevent embedding out-of-bounds crash
            x_cat_ord_val = np.clip(x_cat_ord_val, 0, None)
            cat_cards = [len(c) for c in oe.categories_]
        else:
            x_cat_ohe_train = x_cat_ohe_val = np.zeros((len(train_idx), 0))
            x_cat_ord_train = x_cat_ord_val = np.zeros((len(train_idx), 0))

        x_comb_train = np.hstack([x_num_train, x_cat_ohe_train]) if (cat_cols or num_cols) else x_num_train
        x_comb_val = np.hstack([x_num_val, x_cat_ohe_val]) if (cat_cols or num_cols) else x_num_val

        train_ds = UniversalTabularDataset(x_num_train, x_cat_ord_train, x_comb_train, y[train_idx])
        val_ds = UniversalTabularDataset(x_num_val, x_cat_ord_val, x_comb_val, y[val_idx])

        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

        if model_name == "Basic Tabular NN":
            model = BasicTabularNN(input_dim=x_comb_train.shape[1]).to(DEVICE)
        elif model_name == "Entity Embedding NN":
            model = EntityEmbeddingNN(len(num_cols), cat_cards).to(DEVICE)
        elif model_name == "FT-Transformer":
            model = FTTransformer(len(num_cols), cat_cards).to(DEVICE)
        else:
            raise ValueError(f"Unknown model_name: {model_name}")

        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
        criterion = nn.CrossEntropyLoss()

        model.train()
        for _ in range(epochs):
            for x_n, x_c, x_cb, y_b in train_loader:
                x_n, x_c, x_cb, y_b = x_n.to(DEVICE), x_c.to(DEVICE), x_cb.to(DEVICE), y_b.to(DEVICE)
                optimizer.zero_grad()
                loss = criterion(model(x_n, x_c, x_cb), y_b)
                loss.backward()
                optimizer.step()

        model.eval()
        val_preds = []
        with torch.no_grad():
            for x_n, x_c, x_cb, _ in val_loader:
                x_n, x_c, x_cb = x_n.to(DEVICE), x_c.to(DEVICE), x_cb.to(DEVICE)
                logits = model(x_n, x_c, x_cb)
                val_preds.extend(logits.argmax(dim=1).cpu().numpy())

        fold_scores.append(f1_score(y[val_idx], np.array(val_preds), average="macro"))

    fold_scores = np.array(fold_scores)

    result = {
        "Model": model_name,
        "Experiment": experiment_name,
        "Mean Macro F1": fold_scores.mean(),
        "Std Macro F1": fold_scores.std(),
        "Fold 1": fold_scores[0],
        "Fold 2": fold_scores[1],
        "Fold 3": fold_scores[2],
        "Fold 4": fold_scores[3],
        "Fold 5": fold_scores[4],
    }

    print(
        f"[{model_name:<20}] {experiment_name:<50} "
        f"Macro-F1 = {fold_scores.mean():.4f} (± {fold_scores.std():.4f})"
    )

    return result


In [ ]:
results = []

dataset_items = list(DATASETS.items())[:1] if QUICK_MODE else DATASETS.items()

for dataset_name, original_df in dataset_items:

    print("\n")
    print("=" * 90)
    print(f"DATASET: {dataset_name}")
    print("=" * 90)

    # --------------------------------------------------------
    # 1. GIVEN DATASET / BASELINE
    # --------------------------------------------------------

    df_exp = original_df.copy()

    for model_name in NN_MODELS:
        results.append(
            evaluate_nn_model(
                df=df_exp,
                numeric_features=BASE_NUMERIC_FEATURES,
                categorical_features=BASE_CATEGORICAL_FEATURES,
                model_name=model_name,
                experiment_name=f"{dataset_name} | 1. Baseline"
            )
        )

    # --------------------------------------------------------
    # 2. TRANSFORM SKEWED VARIABLES
    # --------------------------------------------------------

    df_exp = add_skew_transformations(df_exp)

    for model_name in NN_MODELS:
        results.append(
            evaluate_nn_model(
                df=df_exp,
                numeric_features=SKEW_NUMERIC_FEATURES,
                categorical_features=BASE_CATEGORICAL_FEATURES,
                model_name=model_name,
                experiment_name=f"{dataset_name} | 2. Skew transformations"
            )
        )

    if QUICK_MODE:
        continue

    # --------------------------------------------------------
    # 3. CATEGORIZE WEATHER FEATURES
    # --------------------------------------------------------

    df_exp = add_weather_categories(df_exp)

    for model_name in NN_MODELS:
        results.append(
            evaluate_nn_model(
                df=df_exp,
                numeric_features=BASE_NUMERIC_FEATURES + ["Wind speed_log1p"],
                categorical_features=WEATHER_CATEGORY_FEATURES,
                model_name=model_name,
                experiment_name=f"{dataset_name} | 3. + Weather categories"
            )
        )

    # --------------------------------------------------------
    # 4. ADD WEEKDAY
    # --------------------------------------------------------

    df_exp = add_weekday(df_exp)

    categorical_with_weekday = BASE_CATEGORICAL_FEATURES + ["Weekday"]

    for model_name in NN_MODELS:
        results.append(
            evaluate_nn_model(
                df=df_exp,
                numeric_features=SKEW_NUMERIC_FEATURES,
                categorical_features=categorical_with_weekday,
                model_name=model_name,
                experiment_name=f"{dataset_name} | 4. + Weekday"
            )
        )

    # --------------------------------------------------------
    # 5. TRANSFORM TIME VARIABLES TO CYCLIC
    # --------------------------------------------------------

    df_exp = add_weekday(df_exp)
    df_exp = add_month(df_exp)
    df_exp = add_all_cyclic_time_features(df_exp)

    numeric_cyclic = SKEW_NUMERIC_FEATURES + [
        "Weekday_sin", "Weekday_cos",
        "Month_sin", "Month_cos",
        "Hour_sin", "Hour_cos",
        "Season_sin", "Season_cos",
    ]

    categorical_cyclic = ["Holiday", "Functioning Day"]

    for model_name in NN_MODELS:
        results.append(
            evaluate_nn_model(
                df=df_exp,
                numeric_features=numeric_cyclic,
                categorical_features=categorical_cyclic,
                model_name=model_name,
                experiment_name=f"{dataset_name} | 5. Cyclic time: Hour+Weekday+Month+Season"
            )
        )


In [ ]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    "Mean Macro F1",
    ascending=False
).reset_index(drop=True)

display(
    results_df[
        ["Model", "Experiment", "Mean Macro F1", "Std Macro F1"]
    ]
)


In [ ]:
# Same literal-separator fix as tree_models.ipynb (regex=False), plus
# pivot_table's aggfunc instead of the crash-prone .pivot() used in
# logistic_regression.ipynb.
results_table = results_df.copy()

results_table["Dataset"] = results_table["Experiment"].str.split(" | ", regex=False).str[0]
results_table["Step"] = results_table["Experiment"].str.split(" | ", regex=False).str[1]

best_per_dataset_model = (
    results_table
    .loc[
        lambda df: df.groupby(["Dataset", "Model"])["Mean Macro F1"]
        .transform("max") == df["Mean Macro F1"]
    ]
    .sort_values("Mean Macro F1", ascending=False)
)

display(
    best_per_dataset_model[
        ["Dataset", "Model", "Experiment", "Mean Macro F1", "Std Macro F1"]
    ]
)


In [ ]:
best_result = results_df.iloc[0]

print("BEST EXPERIMENT")
print("=" * 50)
print("Model:", best_result["Model"])
print("Experiment:", best_result["Experiment"])
print("Mean Macro F1:", round(best_result["Mean Macro F1"], 4))
print("Std Macro F1:", round(best_result["Std Macro F1"], 4))


In [ ]:
pivot = results_table.pivot_table(
    index=["Dataset", "Model"],
    columns="Step",
    values="Mean Macro F1",
    aggfunc="mean"
)

display(pivot.round(4))
